
# Exp.05 — Quantitative Editing Success, Preservation, and Residual Handling

This experiment is designed for the remaining reviewer requests on motion editing:

1. **Does an explicit pose-code edit actually produce the intended geometric change?**
2. **Are joints outside the edited body part preserved?**
3. **Are frames outside the edited temporal interval preserved?**
4. **What should happen to the residual representation after a pose-code edit?**

The released editing notebook edits the pose code and decodes with residual quantization
dropped. Exp.05 evaluates that released behavior and a controlled alternative:

- **`pose_only`** — edit the pose code and decode from the pose latent only.
- **`retain_residual`** — keep the residual codes extracted from the original motion,
  but replace the pose code with the edited pose code.

No model is retrained.

## Controlled edits

We use four directly measurable pose-code groups:

- left-knee angle
- right-knee angle
- left-elbow angle
- right-elbow angle

The HumanML3D test motions are deterministically sampled and balanced across the four
targets. For each motion, a central temporal window is edited. The target angle category
is shifted by four 10-degree bins from the source median category whenever possible.

This gives an objective, reproducible benchmark without relying on an LLM to interpret
free-form edit instructions.

## Primary metrics

**Edit success**
- strict target-category success rate
- toward-target rate
- reduction in target-category error
- signed angle progress in degrees

**Preservation / locality**
- MPJPE outside the edited temporal window
- MPJPE of non-target joints inside the edited window
- MPJPE of target-limb joints inside the edited window
- target-to-nontarget edit locality ratio

**Residual policy**
- paired comparison of `retain_residual` versus `pose_only`

The comparison is paired at the motion level. Bootstrap 95% confidence intervals are
computed over motions.


In [ ]:

from pathlib import Path
import os, sys, json, hashlib, subprocess, shlex, time, re

PGR = Path(".")
PY = Path("venv/bin/python")
DATA = PGR / "dataset" / "HumanML3D"
PRE = PGR / "revision_preflight"
RESULTS = PGR / "revision_results" / "exp05_editing"
LOGDIR = PGR / "revision_logs" / "exp05_editing"

EXP04B_AUDIT = PGR / "revision_results" / "exp04b_refine" / "exp04b_refine_completion_audit.json"
SCRIPT = PRE / "run_exp05_editing_quant.py"
LOG = LOGDIR / "exp05_editing.log"
PIDFILE = LOGDIR / "exp05_editing.pid"
LAUNCH = LOGDIR / "exp05_editing_launch.json"

for d in [PRE, RESULTS, LOGDIR]:
    d.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

assert PY.exists(), PY
assert PGR.exists(), PGR
assert DATA.exists(), DATA
assert EXP04B_AUDIT.exists(), EXP04B_AUDIT

b = json.loads(EXP04B_AUDIT.read_text())
TOKENIZER_DIR = Path(b["tokenizer_dir"])
TOKENIZER_CKPT = TOKENIZER_DIR / "net_best_fid.pth"

required = [
    TOKENIZER_DIR / "arguments.yaml",
    TOKENIZER_CKPT,
    DATA / "new_joint_vecs",
    DATA / "codes",
    DATA / "test.txt",
    DATA / "Mean.npy",
    DATA / "Std.npy",
    PGR / "models" / "pg_tokenizer.py",
    PGR / "utils" / "codebook.py",
    PGR / "posescript" / "posecodes.py",
]
for p in required:
    assert p.exists(), p

print("=== EXP.05 PRE-FLIGHT ===")
print("Python      :", sys.executable)
print("Expected    :", PY)
print("Tokenizer   :", TOKENIZER_CKPT)
print("Tokenizer SHA256:", sha256(TOKENIZER_CKPT))
print("Data root   :", DATA)
print("[OK] Required editing assets resolved.")


In [ ]:
# Build the standalone Exp.05 experiment script.
script_text = '\nimport os, sys, json, math, time, argparse, hashlib, random\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport yaml\n\nPGR = Path(".")\nos.chdir(PGR)\nsys.path.insert(0, str(PGR))\n\n# NumPy 2.x compatibility for older released code.\nif not hasattr(np, "float"):\n    np.float = float\nif not hasattr(np, "int"):\n    np.int = int\nif not hasattr(np, "bool"):\n    np.bool = bool\n\nfrom models.pg_tokenizer import PoseGuidedTokenizer\nfrom utils.motion_process import recover_from_ric\nfrom utils.codebook import vq_to_range\n\ndef fixseed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for chunk in iter(lambda: f.read(1024*1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\ndef build_tokenizer(folder):\n    import argparse as _argparse\n    cfg = Path(folder) / "arguments.yaml"\n    ckpt_path = Path(folder) / "net_best_fid.pth"\n    a = _argparse.Namespace(**yaml.safe_load(cfg.read_text()))\n\n    net = PoseGuidedTokenizer(\n        a,\n        a.nb_code,\n        a.code_dim,\n        a.output_emb_width,\n        a.down_t,\n        a.stride_t,\n        a.width,\n        a.depth,\n        a.dilation_growth_rate,\n        a.vq_act,\n        a.vq_norm,\n        num_quantizers=a.rvq_num_quantizers,\n        shared_codebook=a.rvq_shared_codebook,\n        quantize_dropout_prob=a.rvq_quantize_dropout_prob,\n        quantize_dropout_cutoff_index=a.rvq_quantize_dropout_cutoff_index,\n        rvq_nb_code=a.rvq_nb_code,\n        mu=a.rvq_mu,\n        residual_ratio=a.rvq_residual_ratio,\n        vq_loss_beta=a.rvq_vq_loss_beta,\n        quantizer_type=a.rvq_quantizer_type,\n        params_soft_ent_loss=a.params_soft_ent_loss,\n        use_ema=(not a.unuse_ema),\n        init_method=a.rvq_init_method,\n    )\n    ckpt = torch.load(ckpt_path, map_location="cpu")\n    net.load_state_dict(ckpt["net"], strict=True)\n    net.cuda().eval()\n    for p in net.parameters():\n        p.requires_grad = False\n    return net, a, ckpt_path, ckpt\n\ndef denorm_motion(x, mean, std):\n    return x * mean.new_tensor(std) + mean.new_tensor(mean.cpu().numpy()) if False else None\n\ndef recover_xyz(motion_norm, mean_np, std_np):\n    """motion_norm: [T,263] tensor on CPU or GPU."""\n    arr = motion_norm.detach().float().cpu().numpy()\n    arr = arr * std_np + mean_np\n    xyz = recover_from_ric(torch.from_numpy(arr).float().unsqueeze(0), 22)\n    return xyz.squeeze(0).cpu().numpy()\n\ndef angle_deg(xyz, triplet):\n    """Geometric angle at the middle joint, in degrees."""\n    a, b, c = triplet\n    v1 = xyz[:, c] - xyz[:, b]\n    v2 = xyz[:, a] - xyz[:, b]\n    n1 = np.linalg.norm(v1, axis=1, keepdims=True)\n    n2 = np.linalg.norm(v2, axis=1, keepdims=True)\n    v1 = v1 / np.clip(n1, 1e-8, None)\n    v2 = v2 / np.clip(n2, 1e-8, None)\n    cosv = np.clip((v1 * v2).sum(axis=1), -1.0, 1.0)\n    return np.degrees(np.arccos(cosv))\n\nANGLE_THRESHOLDS = np.arange(10, 180, 10, dtype=np.float32)\n\ndef angle_category(angles):\n    # categories: <=10 -> 0, <=20 -> 1, ..., >170 -> 17\n    return np.digitize(angles, ANGLE_THRESHOLDS, right=True).astype(np.int64)\n\nTARGETS = [\n    dict(group_id=0, name="left_knee",  triplet=(1,4,7),   target_joints=(1,4,7)),\n    dict(group_id=1, name="right_knee", triplet=(2,5,8),   target_joints=(2,5,8)),\n    dict(group_id=2, name="left_elbow", triplet=(16,18,20), target_joints=(16,18,20)),\n    dict(group_id=3, name="right_elbow",triplet=(17,19,21), target_joints=(17,19,21)),\n]\n\ndef choose_target_cat(source_median, shift_bins):\n    source_median = int(source_median)\n    if source_median <= 8:\n        return min(17, source_median + shift_bins)\n    return max(0, source_median - shift_bins)\n\ndef target_description(cat):\n    if cat == 0:\n        return "<=10 deg"\n    if cat == 17:\n        return ">170 deg (near straight)"\n    lo = cat * 10\n    hi = (cat + 1) * 10\n    return f"({lo},{hi}] deg"\n\ndef edit_pose_tokens(pose_tokens, group_id, tok_start, tok_end, target_cat):\n    out = pose_tokens.copy()\n    g_end, g_start = vq_to_range[group_id]\n    assert (g_end - g_start + 1) == 18\n    out[tok_start:tok_end, g_start:g_end+1] = 0.0\n    out[tok_start:tok_end, g_start + target_cat] = 1.0\n    return out\n\ndef pose_group_categories(pose_tokens, group_id):\n    g_end, g_start = vq_to_range[group_id]\n    block = pose_tokens[:, g_start:g_end+1]\n    if not np.all(block.sum(axis=1) > 0.5):\n        raise RuntimeError(f"Invalid one-hot pose-code block for group {group_id}")\n    return np.argmax(block, axis=1)\n\ndef mpjpe_mm(a, b, frame_mask=None, joint_ids=None):\n    if joint_ids is None:\n        joint_ids = np.arange(a.shape[1])\n    if frame_mask is None:\n        frame_mask = np.ones(a.shape[0], dtype=bool)\n    aa = a[frame_mask][:, joint_ids]\n    bb = b[frame_mask][:, joint_ids]\n    if aa.size == 0:\n        return float("nan")\n    return float(np.linalg.norm(aa - bb, axis=-1).mean() * 1000.0)\n\ndef bootstrap_mean_ci(values, B=1000, seed=2027):\n    x = np.asarray(values, dtype=np.float64)\n    x = x[np.isfinite(x)]\n    if len(x) == 0:\n        return np.nan, np.nan, np.nan\n    rng = np.random.default_rng(seed)\n    means = np.empty(B, dtype=np.float64)\n    for i in range(B):\n        means[i] = rng.choice(x, size=len(x), replace=True).mean()\n    return float(x.mean()), float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975))\n\nparser = argparse.ArgumentParser()\nparser.add_argument("--n-samples", type=int, default=400)\nparser.add_argument("--seed", type=int, default=123)\nparser.add_argument("--shift-bins", type=int, default=4)\nparser.add_argument("--window-frac", type=float, default=0.30)\nparser.add_argument("--max-frames", type=int, default=196)\nparser.add_argument("--min-frames", type=int, default=64)\nparser.add_argument("--tokenizer-dir", type=str, required=True)\nparser.add_argument("--data-root", type=str, default="dataset/HumanML3D")\nparser.add_argument("--out-dir", type=str, required=True)\nparser.add_argument("--bootstrap", type=int, default=1000)\nargs = parser.parse_args()\n\nfixseed(args.seed)\ndevice = torch.device("cuda:0")\nDATA = Path(args.data_root)\nOUT = Path(args.out_dir)\nOUT.mkdir(parents=True, exist_ok=True)\n\nmean_np = np.load(DATA / "Mean.npy").astype(np.float32)\nstd_np = np.load(DATA / "Std.npy").astype(np.float32)\nassert mean_np.shape == (263,), mean_np.shape\nassert std_np.shape == (263,), std_np.shape\n\nnet, net_args, tokenizer_ckpt, tokenizer_state = build_tokenizer(args.tokenizer_dir)\nprint("[OK] Tokenizer loaded:", tokenizer_ckpt)\nprint("[OK] Tokenizer checkpoint iter:", tokenizer_state.get("nb_iter"))\nprint("[OK] Tokenizer SHA256:", sha256(tokenizer_ckpt))\n\nwith open(DATA / "test.txt", "r") as f:\n    test_ids = [x.strip() for x in f if x.strip()]\n\neligible = []\nfor sid in sorted(test_ids):\n    mp = DATA / "new_joint_vecs" / f"{sid}.npy"\n    cp = DATA / "codes" / f"{sid}.npy"\n    if not mp.exists() or not cp.exists():\n        continue\n    try:\n        lm = len(np.load(mp, mmap_mode="r"))\n        lc = len(np.load(cp, mmap_mode="r"))\n    except Exception:\n        continue\n    L = min(lm, lc, args.max_frames)\n    L = (L // 4) * 4\n    if L >= args.min_frames:\n        eligible.append(sid)\n\nrng = np.random.default_rng(args.seed)\neligible = np.asarray(eligible, dtype=object)\nrng.shuffle(eligible)\n\nif len(eligible) < args.n_samples:\n    raise RuntimeError(f"Only {len(eligible)} eligible test motions for requested {args.n_samples}")\n\nselected = eligible[:args.n_samples].tolist()\n(OUT / "exp05_selected_samples.json").write_text(json.dumps({\n    "seed": args.seed,\n    "n_samples": args.n_samples,\n    "selected": selected,\n}, indent=2))\n\nconfig = {\n    "experiment": "Exp05 quantitative controlled editing",\n    "n_samples": args.n_samples,\n    "seed": args.seed,\n    "shift_bins": args.shift_bins,\n    "nominal_shift_deg": args.shift_bins * 10,\n    "window_frac": args.window_frac,\n    "max_frames": args.max_frames,\n    "min_frames": args.min_frames,\n    "targets": TARGETS,\n    "strategies": ["pose_only", "retain_residual"],\n    "tokenizer_checkpoint": str(tokenizer_ckpt),\n    "tokenizer_sha256": sha256(tokenizer_ckpt),\n    "mean_path": str(DATA / "Mean.npy"),\n    "std_path": str(DATA / "Std.npy"),\n}\n(OUT / "exp05_protocol_freeze.json").write_text(json.dumps(config, indent=2))\n\nrows = []\nerror_rows = []\nt0_all = time.time()\n\nfor sidx, sid in enumerate(selected):\n    target = TARGETS[sidx % len(TARGETS)]\n    group_id = target["group_id"]\n\n    try:\n        motion_raw = np.load(DATA / "new_joint_vecs" / f"{sid}.npy").astype(np.float32)\n        pose_full = np.load(DATA / "codes" / f"{sid}.npy").astype(np.float32)\n\n        L = min(len(motion_raw), len(pose_full), args.max_frames)\n        L = (L // 4) * 4\n        motion_raw = motion_raw[:L]\n        pose_tokens = pose_full[:L:4]\n\n        ntok = len(pose_tokens)\n        win = max(4, int(round(ntok * args.window_frac)))\n        win = min(win, max(4, ntok - 2))\n        tok_start = max(1, (ntok - win) // 2)\n        tok_end = min(ntok - 1, tok_start + win)\n        if tok_end <= tok_start:\n            raise RuntimeError(f"Bad edit window: ntok={ntok}, {tok_start}:{tok_end}")\n\n        source_cats = pose_group_categories(pose_tokens, group_id)\n        source_med = int(np.rint(np.median(source_cats[tok_start:tok_end])))\n        target_cat = choose_target_cat(source_med, args.shift_bins)\n\n        edited_tokens = edit_pose_tokens(\n            pose_tokens, group_id, tok_start, tok_end, target_cat\n        )\n\n        motion_norm = ((motion_raw - mean_np) / std_np).astype(np.float32)\n\n        motion_t = torch.from_numpy(motion_norm).float().unsqueeze(0).cuda()\n        pose_t = torch.from_numpy(pose_tokens).float().unsqueeze(0).cuda()\n        edited_t = torch.from_numpy(edited_tokens).float().unsqueeze(0).cuda()\n\n        with torch.no_grad():\n            # Released editing behavior: residual branch dropped.\n            base_pose_norm, *_ = net(\n                code_indices=pose_t,\n                motion=None,\n                drop_out_residual_quantization=True,\n            )\n            edit_pose_norm, *_ = net(\n                code_indices=edited_t,\n                motion=None,\n                drop_out_residual_quantization=True,\n            )\n\n            # Alternative residual policy: retain original residual tokens.\n            _, residual_onehot = net.tokenize(\n                motion=motion_t,\n                code_indices=pose_t,\n                return_one_hot=True,\n            )\n            base_ret_norm = net.inference(\n                residual_codes=residual_onehot,\n                code_indices=pose_t,\n            )\n            edit_ret_norm = net.inference(\n                residual_codes=residual_onehot,\n                code_indices=edited_t,\n            )\n\n        decoded = {\n            "pose_only": (base_pose_norm[0], edit_pose_norm[0]),\n            "retain_residual": (base_ret_norm[0], edit_ret_norm[0]),\n        }\n\n        # Ground truth reconstruction reference from the same RIC representation.\n        gt_xyz = recover_xyz(torch.from_numpy(motion_norm), mean_np, std_np)\n\n        frame_start = tok_start * 4\n        frame_end = min(tok_end * 4, L)\n\n        for strategy, (base_norm, edit_norm) in decoded.items():\n            base_xyz = recover_xyz(base_norm, mean_np, std_np)\n            edit_xyz = recover_xyz(edit_norm, mean_np, std_np)\n\n            T = min(len(base_xyz), len(edit_xyz), len(gt_xyz), L)\n            base_xyz = base_xyz[:T]\n            edit_xyz = edit_xyz[:T]\n            gt_xyz_s = gt_xyz[:T]\n\n            fs = min(frame_start, T)\n            fe = min(frame_end, T)\n            inside = np.zeros(T, dtype=bool)\n            inside[fs:fe] = True\n            outside = ~inside\n\n            target_joints = np.asarray(target["target_joints"], dtype=np.int64)\n            nontarget_joints = np.asarray(\n                [j for j in range(22) if j not in set(target_joints.tolist())],\n                dtype=np.int64,\n            )\n\n            base_ang = angle_deg(base_xyz, target["triplet"])\n            edit_ang = angle_deg(edit_xyz, target["triplet"])\n            base_cat = angle_category(base_ang)\n            edit_cat = angle_category(edit_ang)\n\n            if inside.sum() == 0:\n                raise RuntimeError("Empty inside edit window")\n\n            base_err = np.abs(base_cat[inside] - target_cat)\n            edit_err = np.abs(edit_cat[inside] - target_cat)\n\n            sign = 1.0 if target_cat > source_med else -1.0\n            signed_progress_deg = sign * float(\n                edit_ang[inside].mean() - base_ang[inside].mean()\n            )\n\n            strict_success = float(np.mean(edit_cat[inside] == target_cat))\n            toward_rate = float(np.mean(edit_err < base_err))\n            not_worse_rate = float(np.mean(edit_err <= base_err))\n            cat_error_reduction = float(base_err.mean() - edit_err.mean())\n            direction_success = float(signed_progress_deg > 0)\n\n            outside_mpjpe = mpjpe_mm(edit_xyz, base_xyz, outside, None)\n            inside_non = mpjpe_mm(\n                edit_xyz, base_xyz, inside, nontarget_joints\n            )\n            inside_target = mpjpe_mm(\n                edit_xyz, base_xyz, inside, target_joints\n            )\n            locality_ratio = float(inside_target / max(inside_non, 1e-8))\n            baseline_recon = mpjpe_mm(base_xyz, gt_xyz_s)\n\n            row = {\n                "sample_id": sid,\n                "sample_index": sidx,\n                "strategy": strategy,\n                "target_name": target["name"],\n                "group_id": group_id,\n                "n_frames": T,\n                "n_tokens": ntok,\n                "tok_start": tok_start,\n                "tok_end": tok_end,\n                "frame_start": fs,\n                "frame_end": fe,\n                "source_median_cat": source_med,\n                "target_cat": target_cat,\n                "target_description": target_description(target_cat),\n                "source_target_cat_gap": abs(target_cat - source_med),\n                "strict_success_rate": strict_success,\n                "toward_target_rate": toward_rate,\n                "not_worse_rate": not_worse_rate,\n                "category_error_reduction": cat_error_reduction,\n                "signed_angle_progress_deg": signed_progress_deg,\n                "direction_success": direction_success,\n                "outside_window_mpjpe_mm": outside_mpjpe,\n                "inside_nontarget_mpjpe_mm": inside_non,\n                "inside_target_mpjpe_mm": inside_target,\n                "locality_ratio": locality_ratio,\n                "baseline_recon_mpjpe_mm": baseline_recon,\n            }\n            rows.append(row)\n\n        if (sidx + 1) % 10 == 0 or (sidx + 1) == len(selected):\n            pd.DataFrame(rows).to_csv(OUT / "exp05_per_sample_partial.csv", index=False)\n            elapsed = (time.time() - t0_all) / 60\n            print(\n                f"[RUN] {sidx+1}/{len(selected)} motions "\n                f"({100*(sidx+1)/len(selected):.1f}%) elapsed={elapsed:.1f} min",\n                flush=True,\n            )\n\n    except Exception as e:\n        error_rows.append({\n            "sample_id": sid,\n            "sample_index": sidx,\n            "target_name": target["name"],\n            "error": repr(e),\n        })\n        pd.DataFrame(error_rows).to_csv(OUT / "exp05_errors.csv", index=False)\n        print(f"[ERROR] {sid}: {repr(e)}", flush=True)\n        raise\n\ndf = pd.DataFrame(rows)\nif len(df) != args.n_samples * 2:\n    raise RuntimeError(f"Expected {args.n_samples*2} rows, got {len(df)}")\n\nper_sample = OUT / "exp05_per_sample.csv"\ndf.to_csv(per_sample, index=False)\n\nMETRICS = [\n    "strict_success_rate",\n    "toward_target_rate",\n    "not_worse_rate",\n    "category_error_reduction",\n    "signed_angle_progress_deg",\n    "direction_success",\n    "outside_window_mpjpe_mm",\n    "inside_nontarget_mpjpe_mm",\n    "inside_target_mpjpe_mm",\n    "locality_ratio",\n    "baseline_recon_mpjpe_mm",\n]\n\nsummary_rows = []\nfor strategy in ["pose_only", "retain_residual"]:\n    d = df[df.strategy == strategy]\n    row = {"strategy": strategy, "n": len(d)}\n    for mi, metric in enumerate(METRICS):\n        mean, lo, hi = bootstrap_mean_ci(\n            d[metric].to_numpy(),\n            B=args.bootstrap,\n            seed=2027 + mi,\n        )\n        row[f"{metric}_mean"] = mean\n        row[f"{metric}_ci95_lo"] = lo\n        row[f"{metric}_ci95_hi"] = hi\n    summary_rows.append(row)\n\nsummary = pd.DataFrame(summary_rows)\nsummary.to_csv(OUT / "exp05_summary.csv", index=False)\n\nby_target_rows = []\nfor strategy in ["pose_only", "retain_residual"]:\n    for target_name in [t["name"] for t in TARGETS]:\n        d = df[(df.strategy == strategy) & (df.target_name == target_name)]\n        row = {"strategy": strategy, "target_name": target_name, "n": len(d)}\n        for mi, metric in enumerate(METRICS):\n            mean, lo, hi = bootstrap_mean_ci(\n                d[metric].to_numpy(),\n                B=args.bootstrap,\n                seed=3027 + mi,\n            )\n            row[f"{metric}_mean"] = mean\n            row[f"{metric}_ci95_lo"] = lo\n            row[f"{metric}_ci95_hi"] = hi\n        by_target_rows.append(row)\n\nby_target = pd.DataFrame(by_target_rows)\nby_target.to_csv(OUT / "exp05_by_target.csv", index=False)\n\n# Paired residual-policy difference: retain_residual - pose_only.\nwide = df.pivot(index="sample_id", columns="strategy", values=METRICS)\npaired_rows = []\nfor mi, metric in enumerate(METRICS):\n    x = (\n        wide[(metric, "retain_residual")]\n        - wide[(metric, "pose_only")]\n    ).to_numpy()\n    mean, lo, hi = bootstrap_mean_ci(\n        x, B=args.bootstrap, seed=4027 + mi\n    )\n    paired_rows.append({\n        "metric": metric,\n        "difference": "retain_residual - pose_only",\n        "mean_diff": mean,\n        "ci95_lo": lo,\n        "ci95_hi": hi,\n    })\n\npaired = pd.DataFrame(paired_rows)\npaired.to_csv(OUT / "exp05_residual_policy_paired.csv", index=False)\n\naudit = {\n    **config,\n    "completed_samples": args.n_samples,\n    "rows": len(df),\n    "elapsed_minutes": (time.time() - t0_all) / 60,\n    "per_sample_csv": str(per_sample),\n    "summary_csv": str(OUT / "exp05_summary.csv"),\n    "by_target_csv": str(OUT / "exp05_by_target.csv"),\n    "residual_policy_paired_csv": str(OUT / "exp05_residual_policy_paired.csv"),\n    "errors": len(error_rows),\n}\n(OUT / "exp05_completion_audit.json").write_text(json.dumps(audit, indent=2))\n\nprint("\\n=== EXP.05 SUMMARY ===")\nprint(summary.to_string(index=False))\nprint("\\n=== RESIDUAL POLICY PAIRED DIFFERENCE ===")\nprint(paired.to_string(index=False))\nprint("\\n[OK] EXP05_COMPLETE")\n'
SCRIPT.write_text(script_text)
compile(SCRIPT.read_text(), str(SCRIPT), 'exec')
print('[OK] Script:', SCRIPT)


In [ ]:

# Smoke test: 8 motions, two per target joint.
# This exercises tokenizer loading, pose-only decoding, residual extraction/retention,
# geometry recovery, target-success metrics, and preservation metrics.

SMOKE_DIR = RESULTS / "smoke"
SMOKE_LOG = LOGDIR / "exp05_smoke.log"
SMOKE_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    str(PY), "-u", str(SCRIPT),
    "--n-samples", "8",
    "--seed", "123",
    "--shift-bins", "4",
    "--window-frac", "0.30",
    "--tokenizer-dir", str(TOKENIZER_DIR),
    "--data-root", str(DATA),
    "--out-dir", str(SMOKE_DIR),
    "--bootstrap", "200",
]

print("=== EXP.05 SMOKE ===")
print(" ".join(shlex.quote(x) for x in cmd))
t0 = time.time()

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"

with open(SMOKE_LOG, "w") as f:
    p = subprocess.run(
        cmd,
        cwd=PGR,
        env=env,
        stdout=f,
        stderr=subprocess.STDOUT,
        timeout=7200,
    )

elapsed = (time.time() - t0) / 60
txt = SMOKE_LOG.read_text(errors="replace")

print("returncode:", p.returncode)
print(f"elapsed: {elapsed:.1f} min")
print("\n".join(txt.splitlines()[-160:]))

if p.returncode != 0:
    raise RuntimeError(f"Exp.05 smoke failed: {SMOKE_LOG}")
if "[OK] EXP05_COMPLETE" not in txt:
    raise RuntimeError("Exp.05 smoke returned 0 but completion marker is missing.")

print("[OK] Exp.05 end-to-end smoke passed.")



## Production run

Run the next cell **only after the smoke test prints**

```text
[OK] Exp.05 end-to-end smoke passed.
```

The production experiment uses **400 HumanML3D test motions**, balanced as 100 controlled
edits for each of the four angle targets. This is inference-only and should be much
shorter than Exp.04A/04B training.

Do not relaunch the production cell if the PID is still alive.


In [ ]:

# Launch the 400-motion production experiment in the background.

N_SAMPLES = 400
PROD_DIR = RESULTS / "production"
PROD_DIR.mkdir(parents=True, exist_ok=True)

def pid_alive(pid):
    try:
        os.kill(pid, 0)
        return True
    except Exception:
        return False

old_pid = None
if PIDFILE.exists():
    try:
        old_pid = int(PIDFILE.read_text().strip())
    except Exception:
        pass

old_txt = LOG.read_text(errors="replace") if LOG.exists() else ""

if "[OK] EXP05_COMPLETE" in old_txt:
    print("=== EXP.05 ALREADY COMPLETE ===")
elif old_pid and pid_alive(old_pid):
    print("=== EXP.05 ALREADY RUNNING ===")
    print("PID:", old_pid)
else:
    cmd = [
        str(PY), "-u", str(SCRIPT),
        "--n-samples", str(N_SAMPLES),
        "--seed", "123",
        "--shift-bins", "4",
        "--window-frac", "0.30",
        "--tokenizer-dir", str(TOKENIZER_DIR),
        "--data-root", str(DATA),
        "--out-dir", str(PROD_DIR),
        "--bootstrap", "1000",
    ]

    handle = open(LOG, "w")
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"

    p = subprocess.Popen(
        cmd,
        cwd=PGR,
        env=env,
        stdout=handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    PIDFILE.write_text(str(p.pid))
    LAUNCH.write_text(json.dumps({
        "pid": p.pid,
        "n_samples": N_SAMPLES,
        "command": cmd,
        "log": str(LOG),
        "out_dir": str(PROD_DIR),
        "launched_at_unix": time.time(),
    }, indent=2))

    time.sleep(1)
    print("=== EXP.05 LAUNCH ===")
    print("PID   :", p.pid)
    print("Alive :", pid_alive(p.pid))
    print("Log   :", LOG)
    print("Out   :", PROD_DIR)


In [ ]:

# Read-only production status. Safe to rerun.

def pid_alive(pid):
    try:
        os.kill(pid, 0)
        return True
    except Exception:
        return False

pid = None
if PIDFILE.exists():
    try:
        pid = int(PIDFILE.read_text().strip())
    except Exception:
        pass

partial = RESULTS / "production" / "exp05_per_sample_partial.csv"
done_samples = 0
rows = 0

if partial.exists():
    import pandas as pd
    d = pd.read_csv(partial)
    rows = len(d)
    if "sample_id" in d.columns:
        done_samples = d["sample_id"].nunique()

complete = False
if LOG.exists():
    txt = LOG.read_text(errors="replace")
    complete = "[OK] EXP05_COMPLETE" in txt
else:
    txt = ""

print("=== EXP.05 STATUS ===")
print("PID       :", pid)
print("Alive     :", pid_alive(pid) if pid else False)
print("Motions   :", f"{done_samples}/400")
print("Rows      :", rows, "(expected 800 at completion)")
print("Progress  :", f"{100*done_samples/400:.1f}%")
print("Complete  :", complete)

print("\n=== LOG TAIL ===")
print("\n".join(txt.splitlines()[-100:]))


In [ ]:

# Final reviewer-ready tables. Run after Complete=True.

import json
import pandas as pd

PROD_DIR = RESULTS / "production"
AUDIT = PROD_DIR / "exp05_completion_audit.json"

if not AUDIT.exists():
    raise RuntimeError("Exp.05 production is not complete yet.")

audit = json.loads(AUDIT.read_text())
summary = pd.read_csv(PROD_DIR / "exp05_summary.csv")
by_target = pd.read_csv(PROD_DIR / "exp05_by_target.csv")
paired = pd.read_csv(PROD_DIR / "exp05_residual_policy_paired.csv")

print("=== EXP.05 COMPLETION AUDIT ===")
print(json.dumps(audit, indent=2))

cols = [
    "strategy", "n",
    "strict_success_rate_mean",
    "toward_target_rate_mean",
    "category_error_reduction_mean",
    "signed_angle_progress_deg_mean",
    "outside_window_mpjpe_mm_mean",
    "inside_nontarget_mpjpe_mm_mean",
    "inside_target_mpjpe_mm_mean",
    "locality_ratio_mean",
    "baseline_recon_mpjpe_mm_mean",
]

print("\n=== PRIMARY SUMMARY ===")
display(summary[cols].round(5))

print("\n=== BY TARGET ===")
display(by_target[
    [
        "strategy", "target_name", "n",
        "strict_success_rate_mean",
        "toward_target_rate_mean",
        "outside_window_mpjpe_mm_mean",
        "inside_nontarget_mpjpe_mm_mean",
        "inside_target_mpjpe_mm_mean",
    ]
].round(5))

print("\n=== RESIDUAL POLICY: retain_residual - pose_only ===")
display(paired.round(5))

print("\n[OK] Exp.05 reviewer tables ready.")



## Interpretation rules for the paper

Use the results conservatively.

### Editing success

Higher is better for:

- strict target-category success
- toward-target rate
- category-error reduction
- signed angle progress
- direction success

### Preservation

Lower is better for:

- outside-window MPJPE
- inside-window non-target-joint MPJPE

A large target-limb change together with small non-target change yields a larger
**locality ratio**, which is evidence that the intervention is spatially selective.

### Residual handling

The paired table is `retain_residual - pose_only`.

- negative preservation-error difference → retaining residuals improves preservation
- positive success difference → retaining residuals improves edit success
- the opposite signs indicate a trade-off

Do not claim that the better policy is universally optimal. This experiment evaluates two
fixed inference policies for the trained tokenizer and controlled angle edits.

### Reviewer-facing scope

This experiment directly tests:

- whether pose-code interventions produce the intended geometric effect,
- whether the effect remains localized to the edited body part and temporal interval,
- and whether retaining the pre-edit residual representation helps or conflicts with the edit.

It also provides a stronger empirical answer to the concern that codebook orthogonality
alone does not establish semantic independence.
